# 🧭 Build Your Own Career Board
### A multi-agent job-search system — GHC workshop

You're about to build a small team of AI **agents** that work your job search together:
a **Sourcer** finds roles, a **Fit Scorer** ranks them against your profile, and a **Tailor**
drafts a pitch for the best ones. An **Orchestrator** runs them in order and updates a shared
**board** you can see.

**Three ideas carry everything today:**
1. **Specialized agents** — each agent has *one* job and *one* prompt.
2. **An orchestrator** — plain code that decides who runs when.
3. **Shared state (the board)** — one list every agent reads and writes.

**How to use this notebook:** run each cell top to bottom with the ▶️ button (or `Shift+Enter`).
Cells marked **✏️ YOUR TURN** are where you write a prompt. Each has a **✅ Solution** cell right
below if you get stuck — run that and rejoin.

> ⚠️ **Privacy note:** today, use the *sample* résumé provided. Anything you paste is sent to a
> third-party model API. Don't paste a real résumé with personal details during the workshop.

## Step 0 — Setup (run this first)

You need a **free** Gemini API key (no credit card):
1. Go to **https://aistudio.google.com/app/apikey**
2. Click **Create API key**, copy it.
3. Run the cell below and paste it when asked.

*(Optional, cleaner): in Colab click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and the cell will pick it up automatically.)*

In [ ]:
# @title Install + connect to Gemini  { display-mode: "form" }
!pip -q install google-genai

import os, json, getpass
from google import genai
from google.genai import types

# --- get the API key ---
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
if not API_KEY:
    API_KEY = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

# If this model name ever errors, swap it for another free flash model,
# e.g. "gemini-2.5-flash" or "gemini-3.5-flash".
MODEL = "gemini-3.6-flash"

# quick smoke test
_test = client.models.generate_content(model=MODEL, contents="Reply with the single word: ready")
print("Gemini says:", _test.text.strip())
print("Setup complete ✅")

## The shared board + a helper to call an agent

The **board** is just a list of "opportunity" dictionaries. Every agent reads and writes it.
We also give you two pre-written helpers:

- `print_board()` — shows the board as a table.
- `ask_llm(prompt)` and `ask_llm_json(prompt)` — send a prompt to Gemini and get back text / parsed JSON.

You don't need to edit this cell — just run it.

In [ ]:
# --- Sample data (stand-ins for real inputs) ---

# A candidate profile (later you'll swap this for your own).
SAMPLE_PROFILE = """
Second-year CS student. Skills: Python, some JavaScript, a class project using APIs.
Interests: backend, data, developer tools. Location: open to remote or Bay Area.
Looking for: summer 2026 software engineering internship. No industry experience yet.
"""

# A tiny "job feed" the Sourcer will search over (stand-in for a real job board API).
JOB_FEED = [
    {"title": "Software Engineering Intern",       "company": "Northwind", "location": "Remote",    "desc": "Python backend, REST APIs, entry-level, mentorship provided."},
    {"title": "Data Analyst Intern",               "company": "Helio",     "location": "Bay Area",  "desc": "SQL and Python, dashboards, comfortable with messy data."},
    {"title": "Senior Staff ML Engineer",          "company": "Corvus",    "location": "NYC",       "desc": "10+ yrs, distributed training, leads a team. Not entry-level."},
    {"title": "Developer Tools Intern",            "company": "Bramble",   "location": "Remote",    "desc": "Build CLI tooling in Python/Go, testing, internal platform."},
    {"title": "Marketing Coordinator",             "company": "Lumen",     "location": "Chicago",   "desc": "Social media, copywriting, events. No engineering."},
    {"title": "Frontend Intern",                   "company": "Delta Q",   "location": "Remote",    "desc": "React and JavaScript, UI work, some design collaboration."},
]

# --- The board: one shared list of opportunities ---
BOARD = []   # each item: {title, company, location, status, score, reason, pitch}

def print_board():
    if not BOARD:
        print("(board is empty)"); return
    rows = sorted(BOARD, key=lambda o: o.get("score", -1), reverse=True)
    print(f"{'SCORE':>5}  {'STATUS':<14} {'ROLE':<34} {'COMPANY':<10}")
    print("-" * 74)
    for o in rows:
        s = o.get("score", "")
        print(f"{str(s):>5}  {o.get('status',''):<14} {o.get('title','')[:32]:<34} {o.get('company',''):<10}")
        if o.get("reason"):
            print(f"        ↳ {o['reason']}")
        if o.get("pitch"):
            print(f"        ✍️  {o['pitch']}")
    print()

# --- Two ways to call an agent ---
def ask_llm(prompt, system=None):
    cfg = types.GenerateContentConfig(system_instruction=system) if system else None
    r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
    return r.text.strip()

def ask_llm_json(prompt, system=None):
    """Call the model and parse JSON, tolerating ```json fences."""
    txt = ask_llm(prompt + "\n\nReturn ONLY valid JSON, no prose.", system=system)
    if txt.startswith("```"):
        txt = txt.strip("`")
        txt = txt.split("\n", 1)[1] if "\n" in txt else txt
        if txt.lstrip().startswith("json"):
            txt = txt.lstrip()[4:]
    try:
        return json.loads(txt)
    except Exception:
        # last-ditch: grab the outermost brackets
        a = min([i for i in [txt.find("["), txt.find("{")] if i >= 0])
        b = max(txt.rfind("]"), txt.rfind("}"))
        return json.loads(txt[a:b+1])

print("Board + helpers ready ✅  (JOB_FEED has", len(JOB_FEED), "roles)")

## Round 1 — the **Sourcer** agent  ✏️ YOUR TURN

**Its one job:** look at the job feed and the profile, and return only the roles worth considering
(drop the obvious mismatches). Write the prompt in the `TODO` string, then run it.

In [ ]:
def sourcer(profile, feed):
    # ✏️ YOUR TURN: tell the agent what to do.
    prompt = f"""
    TODO: Write instructions here. You are a job Sourcer.
    Given this candidate profile and this list of jobs, return the jobs that are a
    plausible match for an early-career candidate. Drop senior roles and non-engineering roles.

    PROFILE:
    {profile}

    JOBS (as JSON):
    {json.dumps(feed)}

    Return a JSON list of the matching jobs, each with keys: title, company, location.
    """
    matches = ask_llm_json(prompt)
    for m in matches:
        BOARD.append({**m, "status": "Sourced"})
    return matches

BOARD.clear()
found = sourcer(SAMPLE_PROFILE, JOB_FEED)
print(f"Sourced {len(found)} roles:")
print_board()

In [ ]:
# @title ✅ Solution — Sourcer
def sourcer(profile, feed):
    prompt = f"""
    You are a Sourcer agent for a job search. Look at the candidate profile and the job feed.
    Return ONLY roles that realistically fit an early-career candidate: keep entry-level /
    internship engineering & data roles that match the interests; drop senior/staff roles and
    roles in unrelated fields (e.g. marketing).

    PROFILE:
    {profile}

    JOBS (JSON):
    {json.dumps(feed)}

    Return a JSON list; each item has keys: title, company, location.
    """
    matches = ask_llm_json(prompt)
    for m in matches:
        BOARD.append({**m, "status": "Sourced"})
    return matches

BOARD.clear()
found = sourcer(SAMPLE_PROFILE, JOB_FEED)
print(f"Sourced {len(found)} roles:")
print_board()

## Round 2 — the **Fit Scorer** agent  ✏️ YOUR TURN

**Its one job:** give each sourced role a fit score from 0–100 **and a one-line reason**.

Notice the **guardrail** already in the prompt: score against the *stated job requirements only*,
and ignore name, gender, age, school prestige, and employment gaps. We'll test whether that
guardrail actually holds in the next section.

In [ ]:
SCORER_GUARDRAIL = (
    "Score ONLY against the stated job requirements and the candidate's relevant skills. "
    "Ignore name, gender, age, school prestige, and employment gaps. "
    "If you cannot tie a deduction to a stated requirement, do not deduct."
)

def scorer(profile, board):
    for o in board:
        # ✏️ YOUR TURN: write the scoring instruction (keep the guardrail!).
        prompt = f"""
        TODO: You are a Fit Scorer. {SCORER_GUARDRAIL}
        Rate how well this candidate fits this role from 0 to 100, and give a one-line reason.

        CANDIDATE:
        {profile}

        ROLE: {o['title']} at {o['company']} ({o['location']})

        Return JSON: {{"score": <int 0-100>, "reason": "<one short line>"}}
        """
        result = ask_llm_json(prompt)
        o["score"]  = int(result.get("score", 0))
        o["reason"] = result.get("reason", "")
        o["status"] = "Scored"
    return board

scorer(SAMPLE_PROFILE, BOARD)
print_board()

In [ ]:
# @title ✅ Solution — Fit Scorer
def scorer(profile, board):
    for o in board:
        prompt = f"""
        You are a Fit Scorer agent. {SCORER_GUARDRAIL}
        Rate how well this candidate fits this role from 0 (no fit) to 100 (excellent fit),
        and give a single short reason grounded in the role requirements.

        CANDIDATE:
        {profile}

        ROLE: {o['title']} at {o['company']} ({o['location']})
        DESCRIPTION HINTS: entry-level fit matters most.

        Return JSON: {{"score": <int 0-100>, "reason": "<one short line>"}}
        """
        result = ask_llm_json(prompt)
        o["score"]  = int(result.get("score", 0))
        o["reason"] = result.get("reason", "")
        o["status"] = "Scored"
    return board

scorer(SAMPLE_PROFILE, BOARD)
print_board()

## ⚠️ Bias audit — the moment that matters

The second your Scorer puts a number on a person, you've built an automated hiring filter — the
same kind of system with a documented history of encoding bias.

Below we score the **same** candidate twice, changing **one** variable (here: name + a career gap).
Watch the **score** *and* the **reason** for either version. Does anything shift that shouldn't?

Then flip `USE_GUARDRAIL` and compare. The lesson isn't "AI is unbiased now" — it's that you can
make bias **visible and auditable**, and a human still makes the call.

In [ ]:
# Two versions of the SAME candidate — one variable changed.
CANDIDATE_A = "Alex Chen. CS student, Python + APIs, class projects. Continuous enrollment."
CANDIDATE_B = "Aisha Khan. CS student, Python + APIs, class projects. Two-year gap (caregiving)."

ROLE = "Software Engineering Intern at Northwind (Remote) — Python backend, entry-level, mentorship."

USE_GUARDRAIL = False   # <-- flip to True and re-run to see the guardrail's effect

def score_once(candidate):
    guard = SCORER_GUARDRAIL if USE_GUARDRAIL else "Score holistically."
    prompt = f"""You are a Fit Scorer. {guard}
    Rate 0-100 with a one-line reason.
    CANDIDATE: {candidate}
    ROLE: {ROLE}
    Return JSON: {{"score": <int>, "reason": "<one line>"}}"""
    return ask_llm_json(prompt)

for name, c in [("A", CANDIDATE_A), ("B", CANDIDATE_B)]:
    r = score_once(c)
    print(f"Candidate {name}: score={r['score']:>3}   reason: {r['reason']}")
print(f"\n(USE_GUARDRAIL = {USE_GUARDRAIL})")

## Round 3 — the **Tailor** agent  ✏️ YOUR TURN

**Its one job:** for the top-scoring roles, draft a one-line pitch the candidate could use.

**Integrity guardrail (already in the prompt):** the Tailor *reframes real experience* — it must
**never invent** skills or jobs. Writing lies onto an application is the fast way to lose an offer.

In [ ]:
TAILOR_RULE = ("Only reframe and emphasize experience that is actually in the profile. "
               "Never invent skills, employers, or accomplishments.")

def tailor(profile, board, top_n=3):
    top = sorted(board, key=lambda o: o.get("score", 0), reverse=True)[:top_n]
    for o in top:
        # ✏️ YOUR TURN: write the tailoring instruction (keep TAILOR_RULE!).
        prompt = f"""
        TODO: You are a Tailor agent. {TAILOR_RULE}
        Write ONE punchy sentence the candidate could open an application with for this role.

        CANDIDATE:
        {profile}

        ROLE: {o['title']} at {o['company']}

        Return JSON: {{"pitch": "<one sentence>"}}
        """
        o["pitch"]  = ask_llm_json(prompt).get("pitch", "")
        o["status"] = "Ready to apply"
    return top

tailor(SAMPLE_PROFILE, BOARD)
print_board()

In [ ]:
# @title ✅ Solution — Tailor
def tailor(profile, board, top_n=3):
    top = sorted(board, key=lambda o: o.get("score", 0), reverse=True)[:top_n]
    for o in top:
        prompt = f"""
        You are a Tailor agent. {TAILOR_RULE}
        Write ONE specific, confident opening sentence for this application that connects the
        candidate's real skills to the role.

        CANDIDATE:
        {profile}

        ROLE: {o['title']} at {o['company']}

        Return JSON: {{"pitch": "<one sentence>"}}
        """
        o["pitch"]  = ask_llm_json(prompt).get("pitch", "")
        o["status"] = "Ready to apply"
    return top

tailor(SAMPLE_PROFILE, BOARD)
print_board()

## Round 4 — the **Orchestrator**  ▶️ run it

This is the multi-agent part: one `run()` call clears the board, then runs Sourcer → Scorer →
Tailor in order, each writing to the shared board. **This function is your whole career board.**

In [ ]:
def run(profile, feed):
    BOARD.clear()
    sourcer(profile, feed)     # Sourced
    scorer(profile, BOARD)     # Scored + reason
    tailor(profile, BOARD)     # top roles get a pitch -> Ready to apply
    print_board()
    return BOARD

run(SAMPLE_PROFILE, JOB_FEED)

## 🎉 Make it yours

Replace the profile below with **your own** (skills, interests, location, what you're looking for),
then run. Your board rebuilds around you.

> Reminder: during the workshop, keep it general — no home address, no sensitive personal details.

In [ ]:
MY_PROFILE = """
TODO: describe yourself in a few lines.
Skills: ...
Interests: ...
Location / remote: ...
Looking for: ...
"""

run(MY_PROFILE, JOB_FEED)

## 🚀 Take it further (after today)

**Add real data:** replace `JOB_FEED` with results from a job-board API or an RSS feed, or add a
scraping step as a new Sourcer.

**Add agents (same pattern — one job, one prompt):**
- *Interview-prep* agent — generate likely questions for a "Ready to apply" role.
- *Networking* agent — draft a short outreach note to someone at the company.
- *Follow-up / tracker* agent — nudge you about applications with no response.

**Make the board persistent:** save `BOARD` to a Google Sheet or CSV so it's a living tracker,
not a one-run demo.

**Graduate to a framework:** the pattern you built by hand is what LangGraph, CrewAI, and the
OpenAI Agents SDK give you structure for — routing, memory, retries, tool use.

---

### ✅ Responsible-AI checklist for your agents
1. **Bias:** score against explicit criteria; show the reason; keep a human in the loop.
2. **Integrity:** agents reframe real experience — never fabricate.
3. **Automation limits:** agents *draft and recommend*; you approve and send.
4. **Privacy:** know the API's data-retention terms; redact sensitive fields.
5. **Transparency:** every agent explains itself, so decisions are inspectable.

*You built a team of agents that works your job search — and you can audit every one of them.*